# 🛡️ Tank duel — PPO + GRU self-play on Colab
Train here (GPU for the network, all vCPUs for the arenas), duel on your laptop with `referee.py`.
Runtime → *Change runtime type* → **A100 GPU**. Run the cells top to bottom; after a disconnect run them again — training **resumes** from Google Drive.

In [ ]:
import os, multiprocessing
print("vCPU:", multiprocessing.cpu_count())
!nvidia-smi -L

## 1. Drive (checkpoints survive disconnects) and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # only for checkpoints/exports (they survive disconnects)

RUN_DIR = '/content/drive/MyDrive/tank_runs/run4_v12'   # NEW name = fresh run (rules v1.2: new arena + new network input, old runs do not load); the SAME name = --resume continues it
SOURCE = 'upload'   # 'upload': pick tank_code_v12.zip (both repos at rules v1.2, made locally by `private/make_colab_zip.sh`) | 'git': clone the repos below (they must already contain v1.2)
ARENA_URL = 'https://github.com/BohatyrovIvanUcu/tank_battle_playground.git'   # shared arena + referee (git mode only)
ARENA_REF = 'main'                                   # git mode only, e.g. the tag 'rules-v1.2'
TRAIN_URL = 'https://github.com/partum55/tank-game.git'   # PPO + GRU trainer (git mode only)
os.makedirs(RUN_DIR, exist_ok=True)

In [ ]:
%cd /content
!rm -rf playground tank-game
if SOURCE == 'upload':
    import zipfile
    from google.colab import files
    uploaded = files.upload()                 # choose tank_code_v12.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall('/content')
else:
    os.system(f'git clone -q {ARENA_URL} playground && cd playground && git checkout -q {ARENA_REF}')
    os.system(f'git clone -q {TRAIN_URL} tank-game')
!grep -n "RULES_VERSION =" /content/playground/python/tank_env/__init__.py   # must say 1.2
!ls /content/playground /content/tank-game

## 2. Build the Rust arena (headless: no window libraries needed)

In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ['PATH'] += ':/root/.cargo/bin'
!pip install -q maturin
!maturin build --release --no-default-features -q -o /content/wheels
# --no-deps: Colab already has numpy/torch; letting pip touch numpy breaks them ("numpy.dtype size changed")
!pip install -q --no-deps --force-reinstall /content/wheels/tank_arena-*.whl

In [ ]:
os.environ['PYTHONPATH'] = '/content/playground/python:/content/tank-game'
%cd /content/tank-game
!python -c "import numpy, torch, tank_arena, tank_env; print('numpy', numpy.__version__, '| torch', torch.__version__, '| rules', tank_env.RULES_VERSION); assert tank_env.RULES_VERSION == '1.2', 'wrong arena version'"
!python train_ppo.py --self-check
# the same tests I ran locally (arena parity with the Rust engine, rewards, adaptation); ~1 min. Stop here if anything fails.
!pip install -q pytest
!python -m pytest tests -q -x

## 3. Train
On a GPU runtime the arenas run **on the GPU** (`--env gpu` is chosen automatically: thousands of games in parallel, the Rust engine only builds the maps and does the evaluation). Without a GPU it falls back to the Rust engine on the CPU workers.
Log columns (rules v1.2): `kill / died / timeout` = learner vs hall-of-fame + bot over the last updates (only a **kill** is a win; a timeout with nobody dead is a failure like dying), `bot kill` = vs the rule-based anchor, `shots` = shots fired per episode (the magazine is 20), `hit` = share of shots that hit, `dry` = share of episodes that ended with an empty magazine and no kill, `EVAL vs_bot / vs_old` = fixed maps on the **Rust** engine, both sides (share of games WON: 1.0 = always wins, a draw counts 0), `*best` = new best export. Watch `sps`: it should be tens of thousands or more on the GPU env.
If `shots` collapses to ~0 early the bad-shot penalties scared it off shooting: raise `--shot-penalty-warmup` (default 0.1 = penalties grow from 0 to full over the first 10% of `--steps`).
Raise `--steps` (e.g. `300e6`) for a longer run; continuing a run with `--resume` and the same `RUN_DIR` is safe.

**GPU sizing is automatic**: the trainer benchmarks 1k, 2k, 4k ... arenas, finds where the GPU stops getting faster, and then caps the number of arenas so the run keeps >= 400 PPO updates (PPO learns per update - a giant batch with only a handful of updates learns *worse*). So a bigger `--steps` budget = more arenas = more of the GPU used. The first lines of the log show the benchmark table and the choice; the `vram` column shows memory use. Force a size with `--gpu-envs N`.

**Measured on an A100 (before v1.2):** a simulator step takes ~37 ms whether it runs 1k or 65k arenas (it is launch-bound), so more arenas are almost free - what limits learning is the number of PPO *updates*. Hence a short rollout and many arenas: `--rollout 32 --steps 600e6` -> ~32k arenas, ~570 updates. The v1.2 arena has more kernels per step (SAT collisions, bullet threat), so expect a somewhat slower step - the log prints the real numbers.
Arenas are a mix of the official maps and extra training-only layouts (`--map-mix 0.3`); evaluation always uses the official Rust maps.
**Continuing a run (`--resume`, same `RUN_DIR`) is safe, also with different `--steps`/`--rollout`/arena counts.** A new `RUN_DIR` starts from scratch.

**CUDA graph (`--cuda-graph auto`):** the whole rollout step is recorded once and replayed with a single launch. At start the log prints one line, e.g. `rollout: CUDA graph ON: 37.0 ms/step eager -> 3.1 ms/step replayed (11.9x)` - it times both modes itself and falls back to the plain loop if anything goes wrong. If the run dies with a CUDA error right after the capture, restart the training cell with `--cuda-graph off`.

**New in v1.2 (arena AND inputs changed => start a NEW `RUN_DIR`):**
- arena: **20 shots per tank per match**, **rectangular 40 x 26 tanks** (bullets, walls, tank-tank, lidar and the renderer all use the rectangle);
- reward: **a win is only a kill** (timeout / being killed / both dead = -100), penalties for shooting without seeing the enemy, off the lead point or into a wall, a bonus for consecutive hits, an incentive to find the enemy and a penalty for hiding in corners;
- network input 36: + own ammo, the nearest incoming enemy bullet (urgency, miss distance, side), probability that the enemy is shooting;
- exported agents can **adapt before each match of a series** (`pick_best.py --adapt move,shot,bc,rl`): see TRAINING.md.

In [ ]:
%cd /content/tank-game
!python train_ppo.py --save-dir {RUN_DIR} --steps 1000e6 --rollout 32 --eval-every 20 --snapshot-every 6 --resume

## 4. Hand-off: one zip to download
Training keeps `RUN_DIR/bundle/` fresh at every evaluation (~15 MB: config, per-update metrics, the best and latest agents, a few hall-of-fame snapshots).
Run the cell below (also fine while a session is alive after an interrupt) and send the downloaded **`tank_bundle.zip`**; assess it with

```bash
cd tank-game && PYTHONPATH=../playground/python:. python3 assess.py tank_bundle.zip
```
The verdict says whether the agent is still improving (keep training with `--resume`), plateaued, or stuck. The agent itself is `export_best/` inside the zip (NumPy only, duel it with the shared `referee.py`).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/tank_bundle', 'zip', RUN_DIR + '/bundle')
files.download('/content/tank_bundle.zip')